# 02 — Veri Ön İşleme

**Amaç:** Data leakage'ı önleyerek doğru bir ön işleme pipeline'ı doğrulamak.

**Kurallar:**
- `FILENAME`, `URL`, `Domain`, `Title` → DROP (tanımlayıcı → leakage)
- `TLD` → LabelEncoder (sadece train'e fit)
- StandardScaler → sadece train'e fit, test'e transform
- Train/Test: %80/%20, stratify=y, random_state=42

## 0. Colab Kurulumu

In [ ]:
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_DIR = '/content/phishing-url-detection'
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/erenterakye/phishing-url-detection.git {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull

    %cd {REPO_DIR}
    !pip install -q -r requirements.txt

    DATA_PATH = '/content/drive/MyDrive/PhiUSIIL_Phishing_URL_Dataset.csv'
else:
    REPO_DIR = '..'
    DATA_PATH = '../data/PhiUSIIL_Phishing_URL_Dataset.csv'

sys.path.insert(0, REPO_DIR)
print(f'IN_COLAB={IN_COLAB} | DATA_PATH={DATA_PATH}')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import (
    load_data, drop_identifier_columns, encode_tld,
    scale_features, split_data
)

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Veri Yükleme

In [ ]:
df = load_data(DATA_PATH)
print(f'Ham veri: {df.shape}')

## 2. Tanımlayıcı Sütunları Sil

In [ ]:
df = drop_identifier_columns(df)
print(f'Temizlenmiş veri: {df.shape}')

## 3. Train/Test Bölünmesi (ÖNCE böl, SONRA encode/scale)

In [ ]:
y = df['label']
X = df.drop(columns=['label'])

X_train, X_test, y_train, y_test = split_data(X, y)
print(f'Train sınıf dağılımı:\n{y_train.value_counts()}')
print(f'\nTest sınıf dağılımı:\n{y_test.value_counts()}')

## 4. TLD Encoding (sadece train'e fit)

In [ ]:
X_train, tld_encoder = encode_tld(X_train, fit=True)
X_test, _ = encode_tld(X_test, fit=False, encoder=tld_encoder)
print(f'TLD encode sonrası dtype: {X_train["TLD"].dtype}')
print(f'Benzersiz TLD sayısı: {len(tld_encoder.classes_)}')

## 5. StandardScaler (sadece train'e fit)

In [ ]:
X_train_sc, X_test_sc, scaler = scale_features(X_train, X_test)

check = pd.DataFrame({
    'mean (~0)': X_train_sc.mean()[:5].round(4).values,
    'std (~1)':  X_train_sc.std()[:5].round(4).values,
}, index=X_train_sc.columns[:5])
print('Train skalası kontrolü:')
print(check)

## 6. Özet

In [ ]:
summary = pd.DataFrame([
    ['Tanımlayıcı sütunlar', 'FILENAME, URL, Domain, Title — DROP'],
    ['TLD encoding', 'LabelEncoder — sadece train\'e fit'],
    ['Scaling', 'StandardScaler — sadece train\'e fit'],
    ['Train/Test', '%80/%20, stratify, random_state=42'],
    ['Data leakage', 'YOK'],
    ['Train boyutu', f'{X_train_sc.shape[0]:,} örnek × {X_train_sc.shape[1]} özellik'],
    ['Test boyutu',  f'{X_test_sc.shape[0]:,} örnek × {X_test_sc.shape[1]} özellik'],
], columns=['Adım', 'Detay'])
print(summary.to_string(index=False))